<a href="https://www.kaggle.com/code/eaganrainmahasin/inverted-index?scriptVersionId=351346077" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 1. Memuat Skorpus Berita
Skorpus berita diambil dari notebook pada praktikum sebelumnya hasi dari pra pemrosesan. Hasil tersebut diambil dalam bentuk CSV, tapi karena kolom terakhir seharusnya berupa list maka diparsing dengan ast.literal_eval. 
Setelah itu, pemasangan antara kolom index dokumen dengan kolom hasil stemmed dilakukan dan disimpan dalam variabel korpus

In [1]:
import pandas as pd
import ast

data = pd.read_csv('/kaggle/input/datasets/eaganrainmahasin/dataset-inverted-csv/dataset_inverted.csv')

data['tokens_stemmed'] = data['tokens_stemmed'].apply(ast.literal_eval)

print(type(data['tokens_stemmed'][0]))
print(data['tokens_stemmed'][0][:10])
print(len(data))
print(data.columns)

<class 'list'>
['sekjen', 'pdip', 'sekaligus', 'sekretaris', 'tim', 'menang', 'nasional', 'tpn', 'ganjar', 'pranowo']
1000
Index(['docID', 'title', 'article_text', 'tokens_stemmed'], dtype='object')


In [2]:
korpus = dict(zip(data['docID'], data['tokens_stemmed']))

# 2. Membangun Inverted Index
Dalam inverted index, setiap term memiliki nilai df (document frequency), di mana itu merupakah berapa banyak kata tersebut muncul di tiap dokumen yang berbeda. Maka dari itu, untuk langkah awal kita perlu membuat kata tersebut menjadi unik. 

In [3]:
from collections import defaultdict

def bangun_inverted_index(korpus):
    index_sementara = defaultdict(set)

    for docID, tokens in korpus.items():
        kata_unik_di_dokumen = set(tokens)
        for term in kata_unik_di_dokumen:
            index_sementara[term].add(docID)

    inverted_index = {}
    for term, set_docID in index_sementara.items():
        daftar_docsID_terurut = sorted(set_docID)
        df_term = len(daftar_docsID_terurut)
        inverted_index[term] = (df_term, daftar_docsID_terurut)
    return inverted_index

inverted_index = bangun_inverted_index(korpus)

print(inverted_index['debat'])

(132, [0, 9, 11, 15, 16, 34, 37, 39, 40, 42, 44, 45, 46, 48, 49, 70, 74, 80, 82, 86, 88, 95, 101, 102, 113, 118, 124, 134, 137, 138, 156, 162, 168, 169, 173, 174, 181, 183, 192, 199, 203, 224, 227, 231, 236, 249, 255, 273, 285, 299, 301, 304, 318, 332, 333, 336, 337, 352, 354, 363, 370, 372, 414, 425, 427, 493, 495, 503, 514, 516, 520, 521, 524, 526, 527, 531, 534, 542, 543, 548, 555, 556, 557, 560, 562, 566, 567, 573, 578, 585, 597, 600, 607, 611, 615, 616, 624, 629, 636, 638, 642, 651, 652, 661, 665, 666, 710, 714, 737, 749, 771, 774, 783, 805, 813, 821, 830, 858, 882, 893, 904, 908, 915, 922, 952, 953, 954, 957, 960, 983, 990, 995])


# 3. Algoritma Intersect

In [4]:
def intersect(postings1, postings2):
    hasil = []
    i, j, = 0, 0
    while i<len(postings1) and j<len(postings2):
        if (postings1[i] == postings2[j]):
            hasil.append(postings1[i])
            i+=1
            j+=1
        elif (postings1[i]<postings2[j]):
            i+=1
        else:
            j+=1
    return hasil

docID_debat = inverted_index['debat'][1]
docID_kpu = inverted_index['kpu'][1]
hasil_intersect = intersect(docID_debat, docID_kpu)
print(f"Dokumen mengandung hasil intersect debat dan KPU: {hasil_intersect}")

Dokumen mengandung hasil intersect debat dan KPU: [0, 9, 11, 15, 16, 34, 37, 70, 74, 82, 86, 88, 95, 101, 102, 113, 118, 124, 138, 156, 162, 168, 169, 173, 174, 181, 183, 192, 199, 203, 227, 249, 255, 285, 301, 332, 337, 354, 363, 370, 493, 495, 514, 516, 520, 524, 526, 531, 534, 542, 543, 548, 555, 556, 557, 560, 562, 566, 567, 573, 578, 585, 597, 600, 611, 615, 616, 624, 629, 636, 638, 642, 651, 661, 665, 666, 714, 737, 771, 774, 783, 805, 813, 821, 830, 858, 882, 893, 904, 908, 915, 922, 952, 953, 954, 957, 960, 983, 990, 995]


# 4. Query Boolean

In [5]:
def OR(postings1, postings2):
    hasil = []
    i, j, = 0, 0
    while i<len(postings1) and j<len(postings2):
        if (postings1[i] == postings2[j]):
            hasil.append(postings1[i])
            i+=1
            j+=1
        elif (postings1[i]<postings2[j]):
            hasil.append(postings1[i])
            i+=1
        else:
            hasil.append(postings2[j])
            j+=1

    hasil.extend(postings1[i:])
    hasil.extend(postings2[j:])
    return hasil

def NOT(postings, semua_docID):
    set_postings = set(postings)
    return sorted([d for d in semua_docID if d not in set_postings])

semua_docID = sorted(korpus.keys())

def get_postings(term):
    if term in inverted_index:
        return inverted_index[term][1]
    return []

def proses_query_boolean(query):
    tokens_query = query.lower().split()
    if 'and' in tokens_query:
        idx = tokens_query.index('and')
        term1, term2 = tokens_query[idx-1], tokens_query[idx+1]
        return intersect(get_postings(term1), get_postings(term2))

    elif 'or' in tokens_query:
        idx = tokens_query.index('or')
        term1, term2 = tokens_query[idx-1], tokens_query[idx+1]
        return OR(get_postings(term1), get_postings(term2))

    elif 'not' in tokens_query:
        idx = tokens_query.index('not')
        term1, term2 = tokens_query[idx-1], tokens_query[idx+1]
        hasil_term2 = get_postings(term2)
        return intersect(get_postings(term1), NOT(hasil_term2, semua_docID))

    else:
        return get_postings(tokens_query[0])

In [6]:
print(proses_query_boolean("debat AND kpu"))
print(proses_query_boolean("prabowo OR ganjar"))
print(proses_query_boolean("debat NOT kpu"))

[0, 9, 11, 15, 16, 34, 37, 70, 74, 82, 86, 88, 95, 101, 102, 113, 118, 124, 138, 156, 162, 168, 169, 173, 174, 181, 183, 192, 199, 203, 227, 249, 255, 285, 301, 332, 337, 354, 363, 370, 493, 495, 514, 516, 520, 524, 526, 531, 534, 542, 543, 548, 555, 556, 557, 560, 562, 566, 567, 573, 578, 585, 597, 600, 611, 615, 616, 624, 629, 636, 638, 642, 651, 661, 665, 666, 714, 737, 771, 774, 783, 805, 813, 821, 830, 858, 882, 893, 904, 908, 915, 922, 952, 953, 954, 957, 960, 983, 990, 995]
[0, 5, 7, 8, 9, 11, 12, 16, 17, 22, 24, 25, 27, 28, 29, 30, 31, 32, 34, 35, 36, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55, 56, 57, 60, 61, 63, 65, 68, 69, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 91, 93, 94, 96, 97, 99, 100, 104, 106, 107, 109, 110, 111, 112, 114, 116, 118, 119, 126, 127, 130, 131, 132, 134, 137, 139, 140, 142, 145, 146, 148, 149, 152, 153, 154, 157, 158, 159, 162, 163, 165, 166, 167, 172, 176, 177, 179, 182, 184, 187, 189, 191, 193, 194, 195, 196, 

# 5. Positional Index

In [7]:
def bangun_positional_index(korpus):
    positional_index = defaultdict(lambda: defaultdict(list))

    for docID, tokens in korpus.items():
        for posisi, term in enumerate(tokens):
            positional_index[term][docID].append(posisi)

    return positional_index

positional_index = bangun_positional_index(korpus)

print(positional_index['debat'][0])

[14, 24, 28, 74, 77, 88, 94, 100]


In [8]:
def cari_frasa(term1, term2, positional_index):
    if term1 not in positional_index or term2 not in positional_index:
        return []

    dokumen_term1 = set(positional_index[term1].keys())
    dokumen_term2 = set(positional_index[term2].keys())
    dokumen_kandidat = dokumen_term1 & dokumen_term2  

    hasil = []
    for docID in sorted(dokumen_kandidat):
        posisi_term1 = positional_index[term1][docID]
        posisi_term2 = positional_index[term2][docID]
        for p in posisi_term1:
            if (p + 1) in posisi_term2:
                hasil.append(docID)
                break 
    return hasil

hasil_frasa = cari_frasa('debat', 'capres', positional_index)
print(f"Dokumen dengan frasa 'debat capres': {hasil_frasa}")

if hasil_frasa:
    docID_contoh = hasil_frasa[0]
    print(data[data['docID'] == docID_contoh]['title'].values[0])

Dokumen dengan frasa 'debat capres': [9, 11, 34, 37, 74, 80, 82, 86, 88, 95, 101, 113, 124, 134, 137, 162, 168, 173, 174, 181, 183, 192, 199, 203, 224, 249, 255, 285, 301, 332, 337, 354, 370, 493, 495, 514, 520, 521, 524, 526, 531, 534, 542, 543, 548, 555, 556, 557, 560, 562, 566, 567, 573, 578, 585, 597, 600, 607, 611, 615, 616, 624, 629, 636, 638, 642, 651, 652, 661, 665, 666, 714, 737, 749, 771, 774, 805, 813, 830, 882, 893, 915, 922, 952, 953, 954, 957, 960, 983, 995]
Prabowo-Gibran Siap Hadapi Debat Pertama Pilpres 2024


# 6. Perbandingan

In [9]:
import sys

def hitung_ukuran_index(index_obj):
    """Estimasi ukuran total struktur data index dalam bytes."""
    total = sys.getsizeof(index_obj)
    for key, value in index_obj.items():
        total += sys.getsizeof(key)
        total += sys.getsizeof(value)
        if isinstance(value, tuple):  
            for item in value:
                total += sys.getsizeof(item)
        elif isinstance(value, dict):  
            for docID, daftar_posisi in value.items():
                total += sys.getsizeof(docID) + sys.getsizeof(daftar_posisi)

    return total

ukuran_inverted = hitung_ukuran_index(inverted_index)
ukuran_positional = hitung_ukuran_index(positional_index)

print(f"Ukuran inverted index (tanpa posisi): {ukuran_inverted / 1024:.2f} KB")
print(f"Ukuran positional index (dengan posisi): {ukuran_positional / 1024:.2f} KB")
print(f"Selisih: {(ukuran_positional - ukuran_inverted) / 1024:.2f} KB "
      f"({(ukuran_positional / ukuran_inverted - 1) * 100:.1f}% lebih besar)")

Ukuran inverted index (tanpa posisi): 3322.44 KB
Ukuran positional index (dengan posisi): 24006.61 KB
Selisih: 20684.17 KB (622.6% lebih besar)


In [10]:
total_entri_inverted = sum(df_val for df_val, _ in inverted_index.values())
total_entri_positional = sum(
    len(posisi) for docs in positional_index.values() for posisi in docs.values()
)

print(f"Total entri docID di inverted index: {total_entri_inverted}")
print(f"Total entri posisi di positional index: {total_entri_positional}")

Total entri docID di inverted index: 141729
Total entri posisi di positional index: 257458


# Analisis
Ukuran inverted index 3322,44 KB sementara ukuran positional index 24006,61 KB. Selisihnya adalah 20684,17 KB (622,6% lebih besar). Hal ini dikarenakan positional index menyimpan posisi dari setiap term pada setiap dokumen. Ini tentunya membebani memori karena adanya tambahan berupa penyimpanan berulang dari setiap index dokumen. <br>
Sementara bila ditinjau dari total entri, inverted dokumen memiliki entri docsID sebanyak 141.729 entri dan positional index punya 257.458 total entri posisi. positional index memiliki jumlah entri lebih banyak dari inverted index karena positional index menyimpan posisi dari tiap term pada tiap dokumen sementara inverted index hanya menyimpan  di dokumen mana saja suatu term muncul. <br>
Positional index tetap diperlukan meskipun ukurannya lebih besar karena inverted index hanya mengetahui di mana kata tersebut disimpan tapi tidak bisa menjawab query dengan makna bersebalahan seperti "debat presiden". <br>